In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Modifica questo percorso con il nome esatto della tua cartella su Drive
data_dir = '/content/drive/MyDrive/S1_InSAR_DEM'

print("File trovati:")
for f in os.listdir(data_dir):
    size_mb = os.path.getsize(os.path.join(data_dir, f)) / (1024*1024)
    print(f"  {f}  ({size_mb:.0f} MB)")

File trovati:
  S1A_20190708_split.zip  (274 MB)
  S1B_20190702_split.zip  (497 MB)


In [ ]:
import zipfile
import os

# Cartella di destinazione locale
os.makedirs('/content/data', exist_ok=True)

zip_files = [f for f in os.listdir(data_dir) if f.endswith('.zip')]

for zf in zip_files:
    zip_path = os.path.join(data_dir, zf)
    print(f"Estrazione di {zf}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/data')
    print(f"  completato")

print("\nContenuto di /content/data:")
for item in os.listdir('/content/data'):
    print(f"  {item}")

Estrazione di S1A_20190708_split.zip...
  completato
Estrazione di S1B_20190702_split.zip...
  completato

Contenuto di /content/data:
  S1A_20190708_split
  __MACOSX
  S1B_20190702_split


In [ ]:
%%bash
wget -q https://step.esa.int/downloads/9.0/installers/esa-snap_sentinel_unix_9_0_0.sh \
     -O /tmp/snap_installer.sh
echo "Download completato"

Download completato


In [ ]:
%%bash
chmod +x /tmp/snap_installer.sh
/tmp/snap_installer.sh -q -dir /opt/snap 2>/dev/null
echo "Installazione completata"

Unpacking JRE ...
Starting Installer ...
The installation directory has been set to /opt/snap.
Extracting files ...
Finishing installation ...
Installazione completata


In [ ]:
%%bash
/opt/snap/bin/gpt --help 2>/dev/null | head -5

In [ ]:
%%bash
ls -lh /tmp/snap_installer.sh

-rwxr-xr-x 1 root root 985M Jun 29  2022 /tmp/snap_installer.sh


In [ ]:
%%bash
find / -name "gpt" 2>/dev/null

/opt/snap/bin/gpt


CalledProcessError: Command 'b'find / -name "gpt" 2>/dev/null\n'' returned non-zero exit status 1.

In [ ]:
%%bash
/opt/snap/bin/gpt --version

INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: GDAL not found on system. Internal GDAL 3.2.1 from distribution will be used. (f0)
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Currently installed 9.0.0, available is 13.0.0.
Please visit http://step.esa.int


Error: Unknown option '--version'


CalledProcessError: Command 'b'/opt/snap/bin/gpt --version\n'' returned non-zero exit status 1.

In [ ]:
%%bash
snaphu --version 2>&1

bash: line 1: snaphu: command not found


CalledProcessError: Command 'b'snaphu --version 2>&1\n'' returned non-zero exit status 127.

In [ ]:
%%bash
apt-get install -y snaphu 2>&1

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  snaphu
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 161 kB of archives.
After this operation, 411 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/multiverse amd64 snaphu amd64 2.0.5-1 [161 kB]
Fetched 161 kB in 0s (895 kB/s)
Selecting previously unselected package snaphu.
(Reading database ... 118212 files and directories currently installed.)
Preparing to unpack .../snaphu_2.0.5-1_amd64.deb ...
Unpacking snaphu (2.0.5-1) ...
Setting up snaphu (2.0.5-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import os

for root, dirs, files in os.walk('/content/data'):
    if '__MACOSX' in root:
        continue
    level = root.replace('/content/data', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024*1024)
        print(f"{indent}  {f}  ({size_mb:.1f} MB)")

data/
  S1A_20190708_split/
    S1A_IW_SLC__1SDV_20190708T032532_20190708T032559_028020_032A14_33CA_split.dim  (2.5 MB)
    S1A_IW_SLC__1SDV_20190708T032532_20190708T032559_028020_032A14_33CA_split.data/
      i_IW2_VV.hdr  (0.0 MB)
      q_IW2_VV.img  (226.0 MB)
      q_IW2_VV.hdr  (0.0 MB)
      i_IW2_VV.img  (226.0 MB)
      tie_point_grids/
        longitude.hdr  (0.0 MB)
        incident_angle.img  (0.0 MB)
        elevation_angle.hdr  (0.0 MB)
        latitude.img  (0.0 MB)
        longitude.img  (0.0 MB)
        elevation_angle.img  (0.0 MB)
        slant_range_time.img  (0.0 MB)
        slant_range_time.hdr  (0.0 MB)
        incident_angle.hdr  (0.0 MB)
        latitude.hdr  (0.0 MB)
      vector_data/
        pins.csv  (0.0 MB)
        ground_control_points.csv  (0.0 MB)
  S1B_20190702_split/
    S1B_IW_SLC__1SDV_20190702T032447_20190702T032514_016949_01FE47_69C5_split.dim  (2.4 MB)
    S1B_IW_SLC__1SDV_20190702T032447_20190702T032514_016949_01FE47_69C5_split.data/
      i_IW2

In [ ]:
import os
import subprocess

# --- TOOL ---
GPT = '/opt/snap/bin/gpt'

# --- CARTELLE ---
DATA_DIR   = '/content/data'
OUTPUT_DIR = '/content/output'
DRIVE_DIR  = '/content/drive/MyDrive/S1_InSAR_DEM'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/graphs', exist_ok=True)

# --- FILE DI INPUT ---
MASTER = '/content/data/S1A_20190708_split/S1A_IW_SLC__1SDV_20190708T032532_20190708T032559_028020_032A14_33CA_split.dim'
SLAVE  = '/content/data/S1B_20190702_split/S1B_IW_SLC__1SDV_20190702T032447_20190702T032514_016949_01FE47_69C5_split.dim'

# --- VERIFICA ---
for label, f in [('MASTER', MASTER), ('SLAVE', SLAVE)]:
    exists = os.path.exists(f)
    print(f"{'✓' if exists else '✗'}  {label}: {os.path.basename(f)}")

✓  MASTER: S1A_IW_SLC__1SDV_20190708T032532_20190708T032559_028020_032A14_33CA_split.dim
✓  SLAVE: S1B_IW_SLC__1SDV_20190702T032447_20190702T032514_016949_01FE47_69C5_split.dim


In [ ]:
def run_gpt(graph_file, description=""):
    """Esegue un graph XML con GPT e mostra l'output."""
    print(f"\n{'='*50}")
    print(f"Avvio: {description}")
    print(f"{'='*50}")

    cmd = [GPT, graph_file, '-q', '4']  # -q 4 = usa 4 thread

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    # Mostra solo le righe rilevanti (no spam di INFO)
    for line in result.stderr.split('\n'):
        if any(x in line for x in ['ERROR', 'WARNING', 'done', 'Done', '100%', 'written']):
            print(line)

    if result.returncode == 0:
        print(f"\n✓ Completato: {description}")
    else:
        print(f"\n✗ ERRORE in: {description}")
        print(result.stderr[-2000:])  # Ultimi 2000 caratteri dell'errore

    return result.returncode == 0

In [ ]:
print("Funzione run_gpt definita correttamente ✓")
print(f"GPT path: {GPT}")
print(f"MASTER esiste: {os.path.exists(MASTER)}")
print(f"SLAVE esiste: {os.path.exists(SLAVE)}")

Funzione run_gpt definita correttamente ✓
GPT path: /opt/snap/bin/gpt
MASTER esiste: True
SLAVE esiste: True


In [ ]:
graph_orbit_master = f"""
<graph id="ApplyOrbit">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>{MASTER}</file>
    </parameters>
  </node>
  <node id="Apply-Orbit-File">
    <operator>Apply-Orbit-File</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <orbitType>Sentinel Precise (Auto Download)</orbitType>
      <polyDegree>3</polyDegree>
      <continueOnFail>true</continueOnFail>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Apply-Orbit-File"/>
    </sources>
    <parameters>
      <file>{OUTPUT_DIR}/S1A_20190708_Orb</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
"""

# Salva il graph su file
graph_path = f'{OUTPUT_DIR}/graphs/orbit_master.xml'
with open(graph_path, 'w') as f:
    f.write(graph_orbit_master)
print(f"Graph salvato: {graph_path}")

Graph salvato: /content/output/graphs/orbit_master.xml


In [ ]:
graph_orbit_slave = f"""
<graph id="ApplyOrbit">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>{SLAVE}</file>
    </parameters>
  </node>
  <node id="Apply-Orbit-File">
    <operator>Apply-Orbit-File</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <orbitType>Sentinel Precise (Auto Download)</orbitType>
      <polyDegree>3</polyDegree>
      <continueOnFail>true</continueOnFail>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Apply-Orbit-File"/>
    </sources>
    <parameters>
      <file>{OUTPUT_DIR}/S1B_20190702_Orb</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
"""

graph_path = f'{OUTPUT_DIR}/graphs/orbit_slave.xml'
with open(graph_path, 'w') as f:
    f.write(graph_orbit_slave)
print(f"Graph salvato: {graph_path}")

Graph salvato: /content/output/graphs/orbit_slave.xml


In [ ]:
run_gpt(f'{OUTPUT_DIR}/graphs/orbit_master.xml',
        "Apply Orbit File - MASTER (S1A 20190708)")


Avvio: Apply Orbit File - MASTER (S1A 20190708)

✓ Completato: Apply Orbit File - MASTER (S1A 20190708)


True

In [ ]:
run_gpt(f'{OUTPUT_DIR}/graphs/orbit_slave.xml',
        "Apply Orbit File - SLAVE (S1B 20190702)")


Avvio: Apply Orbit File - SLAVE (S1B 20190702)

✓ Completato: Apply Orbit File - SLAVE (S1B 20190702)


True

In [ ]:
import os

for name in ['S1A_20190708_Orb', 'S1B_20190702_Orb']:
    dim = f'{OUTPUT_DIR}/{name}.dim'
    data = f'{OUTPUT_DIR}/{name}.data'
    dim_ok = os.path.exists(dim)
    data_ok = os.path.exists(data)
    print(f"{name}:")
    print(f"  {'✓' if dim_ok else '✗'}  .dim")
    print(f"  {'✓' if data_ok else '✗'}  .data/")

S1A_20190708_Orb:
  ✓  .dim
  ✓  .data/
S1B_20190702_Orb:
  ✓  .dim
  ✓  .data/


In [ ]:
graph_backgeocoding = f"""
<graph id="BackGeocoding">
  <version>1.0</version>
  <node id="Read-Master">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>{OUTPUT_DIR}/S1A_20190708_Orb.dim</file>
    </parameters>
  </node>
  <node id="Read-Slave">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>{OUTPUT_DIR}/S1B_20190702_Orb.dim</file>
    </parameters>
  </node>
  <node id="Back-Geocoding">
    <operator>Back-Geocoding</operator>
    <sources>
      <sourceProduct refid="Read-Master"/>
      <sourceProduct refid="Read-Slave"/>
    </sources>
    <parameters>
      <demName>SRTM 1Sec HGT</demName>
      <demResamplingMethod>BILINEAR_INTERPOLATION</demResamplingMethod>
      <resamplingType>BILINEAR_INTERPOLATION</resamplingType>
      <maskOutAreaWithoutElevation>true</maskOutAreaWithoutElevation>
      <outputRangeAzimuthOffset>false</outputRangeAzimuthOffset>
      <outputDerampDemodPhase>false</outputDerampDemodPhase>
      <disableReramp>false</disableReramp>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Back-Geocoding"/>
    </sources>
    <parameters>
      <file>{OUTPUT_DIR}/S1_Stack</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
"""

graph_path = f'{OUTPUT_DIR}/graphs/backgeocoding.xml'
with open(graph_path, 'w') as f:
    f.write(graph_backgeocoding)
print(f"Graph salvato: {graph_path}")

Graph salvato: /content/output/graphs/backgeocoding.xml


In [ ]:
run_gpt(f'{OUTPUT_DIR}/graphs/backgeocoding.xml',
        "Back Geocoding - Coregistrazione")


Avvio: Back Geocoding - Coregistrazione

✗ ERRORE in: Back Geocoding - Coregistrazione
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: GDAL not found on system. Internal GDAL 3.2.1 from distribution will be used. (f0)
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.hsqldb.persist.Logger: dataFileCache open start

Error: [NodeId: Back-Geocoding] Operator 'BackGeocodingOp': Source product not found: S1A_20190708_Orb
-- org.jblas INFO Deleting /tmp/jblas3858946184966878590/libjblas.so
-- org.jblas INFO Deleting /tmp/jblas3858946184966878590/libjblas_arch_flavor.so
-- org.jblas INFO Deleting /tmp/jblas3858946184966878590/libquadma

False

In [ ]:
import os

for name in ['S1A_20190708_Orb', 'S1B_20190702_Orb']:
    dim = f'{OUTPUT_DIR}/{name}.dim'
    size = os.path.getsize(dim) / (1024*1024) if os.path.exists(dim) else 0
    print(f"{'✓' if os.path.exists(dim) else '✗'}  {dim}  ({size:.1f} MB)")

✓  /content/output/S1A_20190708_Orb.dim  (2.5 MB)
✓  /content/output/S1B_20190702_Orb.dim  (2.4 MB)


In [ ]:
graph_backgeocoding = f"""
<graph id="BackGeocoding">
  <version>1.0</version>
  <node id="ProductSet-Reader">
    <operator>ProductSet-Reader</operator>
    <sources/>
    <parameters>
      <fileList>{OUTPUT_DIR}/S1A_20190708_Orb.dim,{OUTPUT_DIR}/S1B_20190702_Orb.dim</fileList>
    </parameters>
  </node>
  <node id="Back-Geocoding">
    <operator>Back-Geocoding</operator>
    <sources>
      <sourceProduct refid="ProductSet-Reader"/>
    </sources>
    <parameters>
      <demName>SRTM 1Sec HGT</demName>
      <demResamplingMethod>BILINEAR_INTERPOLATION</demResamplingMethod>
      <resamplingType>BILINEAR_INTERPOLATION</resamplingType>
      <maskOutAreaWithoutElevation>true</maskOutAreaWithoutElevation>
      <outputRangeAzimuthOffset>false</outputRangeAzimuthOffset>
      <outputDerampDemodPhase>false</outputDerampDemodPhase>
      <disableReramp>false</disableReramp>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Back-Geocoding"/>
    </sources>
    <parameters>
      <file>{OUTPUT_DIR}/S1_Stack</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
"""

graph_path = f'{OUTPUT_DIR}/graphs/backgeocoding.xml'
with open(graph_path, 'w') as f:
    f.write(graph_backgeocoding)
print(f"Graph salvato: {graph_path}")

Graph salvato: /content/output/graphs/backgeocoding.xml


In [ ]:
run_gpt(f'{OUTPUT_DIR}/graphs/backgeocoding.xml',
        "Back Geocoding - Coregistrazione")


Avvio: Back Geocoding - Coregistrazione


KeyboardInterrupt: 

In [ ]:
%%bash
ls -lh /content/output/
ls -lh /content/output/S1_Stack.data/ 2>/dev/null || echo "Nessun output ancora"

total 7.4M
drwxr-xr-x 2 root root 4.0K May 25 19:18 graphs
drwxr-xr-x 4 root root 4.0K May 25 19:14 S1A_20190708_Orb.data
-rw-r--r-- 1 root root 2.5M May 25 19:15 S1A_20190708_Orb.dim
drwxr-xr-x 4 root root 4.0K May 25 19:16 S1B_20190702_Orb.data
-rw-r--r-- 1 root root 2.4M May 25 19:16 S1B_20190702_Orb.dim
drwxr-xr-x 4 root root 4.0K May 25 19:20 S1_Stack.data
-rw-r--r-- 1 root root 2.6M May 25 19:20 S1_Stack.dim
total 8.0K
drwxr-xr-x 2 root root 4.0K May 25 19:20 tie_point_grids
drwxr-xr-x 2 root root 4.0K May 25 19:20 vector_data


In [ ]:
import zipfile
import shutil
import os

# Rimuovi la vecchia cartella S1B
old_path = '/content/data/S1B_20190702_split'
if os.path.exists(old_path):
    shutil.rmtree(old_path)
    print("✓ Vecchio S1B rimosso")

# Estrai il nuovo
zip_path = f'{DRIVE_DIR}/S1B_new_split.zip'
print(f"Estrazione di {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/data')
print("✓ Estrazione completata")

# Verifica struttura
print("\nContenuto di /content/data:")
for root, dirs, files in os.walk('/content/data'):
    if '__MACOSX' in root:
        continue
    level = root.replace('/content/data', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024*1024)
        print(f"{indent}  {f}  ({size_mb:.1f} MB)")

✓ Vecchio S1B rimosso
Estrazione di /content/drive/MyDrive/S1_InSAR_DEM/S1B_new_split.zip...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/S1_InSAR_DEM/S1B_new_split.zip'

In [ ]:
import os

print("Contenuto della cartella Drive:")
for f in os.listdir(DRIVE_DIR):
    size_mb = os.path.getsize(os.path.join(DRIVE_DIR, f)) / (1024*1024)
    print(f"  {f}  ({size_mb:.0f} MB)")

Contenuto della cartella Drive:
  S1A_20190708_split.zip  (274 MB)
  S1B_20190702_split.zip  (497 MB)
  S1B_new_split.zip  (269 MB)


In [ ]:
import zipfile
import os

# Forza il refresh del filesystem di Drive
import importlib
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

# Riprova l'estrazione
zip_path = f'{DRIVE_DIR}/S1B_new_split.zip'
print(f"Estrazione di {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/data')
print("✓ Estrazione completata")

print("\nContenuto di /content/data:")
for item in os.listdir('/content/data'):
    if '__MACOSX' not in item:
        print(f"  {item}")

Mounted at /content/drive
Estrazione di /content/drive/MyDrive/S1_InSAR_DEM/S1B_new_split.zip...
✓ Estrazione completata

Contenuto di /content/data:
  S1A_20190708_split
  S1B_new_split


In [ ]:
import os

# Trova il file .dim del nuovo S1B
for root, dirs, files in os.walk('/content/data/S1B_new_split'):
    if '__MACOSX' in root:
        continue
    for f in files:
        if f.endswith('.dim'):
            new_slave = os.path.join(root, f)
            size_mb = os.path.getsize(new_slave) / (1024*1024)
            print(f"✓ Trovato: {new_slave} ({size_mb:.1f} MB)")

# Aggiorna la variabile SLAVE
SLAVE = new_slave
print(f"\nSLAVE aggiornato: {SLAVE}")

✓ Trovato: /content/data/S1B_new_split/S1B_IW_SLC__1SDV_20190702T032447_20190702T032514_016949_01FE47_69C5_split.dim (2.4 MB)

SLAVE aggiornato: /content/data/S1B_new_split/S1B_IW_SLC__1SDV_20190702T032447_20190702T032514_016949_01FE47_69C5_split.dim


In [ ]:
graph_orbit_slave = f"""
<graph id="ApplyOrbit">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>{SLAVE}</file>
    </parameters>
  </node>
  <node id="Apply-Orbit-File">
    <operator>Apply-Orbit-File</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <orbitType>Sentinel Precise (Auto Download)</orbitType>
      <polyDegree>3</polyDegree>
      <continueOnFail>true</continueOnFail>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Apply-Orbit-File"/>
    </sources>
    <parameters>
      <file>{OUTPUT_DIR}/S1B_20190702_Orb</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
"""

graph_path = f'{OUTPUT_DIR}/graphs/orbit_slave.xml'
with open(graph_path, 'w') as f:
    f.write(graph_orbit_slave)

run_gpt(f'{OUTPUT_DIR}/graphs/orbit_slave.xml',
        "Apply Orbit File - SLAVE nuovo (S1B 20190702)")


Avvio: Apply Orbit File - SLAVE nuovo (S1B 20190702)

✓ Completato: Apply Orbit File - SLAVE nuovo (S1B 20190702)


True

In [ ]:
import shutil

# Rimuovi lo stack incompleto precedente
stack_path = f'{OUTPUT_DIR}/S1_Stack.data'
stack_dim = f'{OUTPUT_DIR}/S1_Stack.dim'
if os.path.exists(stack_path):
    shutil.rmtree(stack_path)
    print("✓ Vecchio S1_Stack.data rimosso")
if os.path.exists(stack_dim):
    os.remove(stack_dim)
    print("✓ Vecchio S1_Stack.dim rimosso")

✓ Vecchio S1_Stack.data rimosso
✓ Vecchio S1_Stack.dim rimosso


In [ ]:
run_gpt(f'{OUTPUT_DIR}/graphs/backgeocoding.xml',
        "Back Geocoding - Coregistrazione (secondo tentativo)")


Avvio: Back Geocoding - Coregistrazione (secondo tentativo)

✓ Completato: Back Geocoding - Coregistrazione (secondo tentativo)


True

In [ ]:
import shutil
import os

def salva_su_drive(nome_file):
    """Copia un prodotto BEAM-DIMAP su Google Drive."""
    src_dim = f'{OUTPUT_DIR}/{nome_file}.dim'
    src_data = f'{OUTPUT_DIR}/{nome_file}.data'
    dst_dim = f'{DRIVE_DIR}/{nome_file}.dim'
    dst_data = f'{DRIVE_DIR}/{nome_file}.data'

    # Copia .dim
    if os.path.exists(src_dim):
        shutil.copy2(src_dim, dst_dim)
        size = os.path.getsize(dst_dim) / (1024*1024)
        print(f"✓ {nome_file}.dim ({size:.1f} MB)")
    else:
        print(f"✗ Non trovato: {nome_file}.dim")

    # Copia cartella .data
    if os.path.exists(src_data):
        if os.path.exists(dst_data):
            shutil.rmtree(dst_data)
        shutil.copytree(src_data, dst_data)
        # Calcola dimensione totale
        size = sum(
            os.path.getsize(os.path.join(r, f))
            for r, d, files in os.walk(dst_data)
            for f in files
        ) / (1024*1024)
        print(f"✓ {nome_file}.data/ ({size:.0f} MB)")
    else:
        print(f"✗ Non trovato: {nome_file}.data/")

# Salva tutti i file importanti su Drive
print("Salvataggio su Google Drive...")
print("="*40)
for nome in ['S1A_20190708_Orb',
             'S1B_20190702_Orb',
             'S1_Stack']:
    print(f"\n→ {nome}")
    salva_su_drive(nome)

print("\n" + "="*40)
print("✓ Salvataggio completato!")

Salvataggio su Google Drive...

→ S1A_20190708_Orb
✓ S1A_20190708_Orb.dim (2.5 MB)
✓ S1A_20190708_Orb.data/ (452 MB)

→ S1B_20190702_Orb
✓ S1B_20190702_Orb.dim (2.4 MB)
✓ S1B_20190702_Orb.data/ (436 MB)

→ S1_Stack
✓ S1_Stack.dim (2.5 MB)
✓ S1_Stack.data/ (1356 MB)

✓ Salvataggio completato!


In [ ]:
import os

print("Verifica file su Drive:")
for nome in ['S1A_20190708_Orb', 'S1B_20190702_Orb', 'S1_Stack']:
    dim = f'{DRIVE_DIR}/{nome}.dim'
    data = f'{DRIVE_DIR}/{nome}.data'
    if os.path.exists(dim) and os.path.exists(data):
        size = sum(
            os.path.getsize(os.path.join(r, f))
            for r, d, files in os.walk(data)
            for f in files
        ) / (1024*1024)
        print(f"✓ {nome} ({size:.0f} MB)")
    else:
        print(f"✗ {nome} - MANCANTE!")

Verifica file su Drive:
✓ S1A_20190708_Orb (452 MB)
✓ S1B_20190702_Orb (436 MB)
✓ S1_Stack (1356 MB)


In [ ]:
%%bash
wget -q https://step.esa.int/downloads/9.0/installers/esa-snap_sentinel_unix_9_0_0.sh \
     -O /tmp/snap_installer.sh
chmod +x /tmp/snap_installer.sh
/tmp/snap_installer.sh -q -dir /opt/snap 2>/dev/null
apt-get install -q -y snaphu 2>/dev/null
echo "✓ SNAP e snaphu installati"

Unpacking JRE ...
Starting Installer ...
The installation directory has been set to /opt/snap.
Extracting files ...
Finishing installation ...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  snaphu
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 161 kB of archives.
After this operation, 411 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/multiverse amd64 snaphu amd64 2.0.5-1 [161 kB]
Fetched 161 kB in 0s (417 kB/s)
Selecting previously unselected package snaphu.
(Reading database ... 118212 files and directories currently installed.)
Preparing to unpack .../snaphu_2.0.5-1_amd64.deb ...
Unpacking snaphu (2.0.5-1) ...
Setting up snaphu (2.0.5-1) ...
Processing triggers for man-db (2.10.2-1) ...
✓ SNAP e snaphu installati


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import subprocess

# Percorsi
GPT        = '/opt/snap/bin/gpt'
OUTPUT_DIR = '/content/output'
DRIVE_DIR  = '/content/drive/MyDrive/S1_InSAR_DEM'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/graphs', exist_ok=True)

# Funzione run_gpt
def run_gpt(graph_file, description=""):
    print(f"\n{'='*50}")
    print(f"Avvio: {description}")
    print(f"{'='*50}")
    cmd = [GPT, graph_file, '-q', '4']
    result = subprocess.run(cmd, capture_output=True, text=True)
    for line in result.stderr.split('\n'):
        if any(x in line for x in ['ERROR', 'WARNING', 'done', 'Done', '100%', 'written']):
            print(line)
    if result.returncode == 0:
        print(f"\n✓ Completato: {description}")
    else:
        print(f"\n✗ ERRORE in: {description}")
        print(result.stderr[-2000:])
    return result.returncode == 0

# Funzione salvataggio Drive
def salva_su_drive(nome_file):
    src_dim = f'{OUTPUT_DIR}/{nome_file}.dim'
    src_data = f'{OUTPUT_DIR}/{nome_file}.data'
    dst_dim = f'{DRIVE_DIR}/{nome_file}.dim'
    dst_data = f'{DRIVE_DIR}/{nome_file}.data'
    if os.path.exists(src_dim):
        shutil.copy2(src_dim, dst_dim)
        print(f"✓ {nome_file}.dim")
    if os.path.exists(src_data):
        if os.path.exists(dst_data):
            shutil.rmtree(dst_data)
        shutil.copytree(src_data, dst_data)
        print(f"✓ {nome_file}.data/")

print("✓ Variabili e funzioni pronte")

✓ Variabili e funzioni pronte


In [ ]:
def ripristina_da_drive(nome_file):
    src_dim = f'{DRIVE_DIR}/{nome_file}.dim'
    src_data = f'{DRIVE_DIR}/{nome_file}.data'
    dst_dim = f'{OUTPUT_DIR}/{nome_file}.dim'
    dst_data = f'{OUTPUT_DIR}/{nome_file}.data'

    ok = True
    if os.path.exists(src_dim):
        shutil.copy2(src_dim, dst_dim)
        print(f"✓ {nome_file}.dim")
    else:
        print(f"✗ {nome_file}.dim - NON TROVATO SU DRIVE")
        ok = False
    if os.path.exists(src_data):
        if os.path.exists(dst_data):
            shutil.rmtree(dst_data)
        shutil.copytree(src_data, dst_data)
        size = sum(
            os.path.getsize(os.path.join(r, f))
            for r, d, files in os.walk(dst_data)
            for f in files
        ) / (1024*1024)
        print(f"✓ {nome_file}.data/ ({size:.0f} MB)")
    else:
        print(f"✗ {nome_file}.data/ - NON TROVATO SU DRIVE")
        ok = False
    return ok

print("Ripristino file da Drive...")
print("="*40)
for nome in ['S1A_20190708_Orb', 'S1B_20190702_Orb', 'S1_Stack']:
    print(f"\n→ {nome}")
    ripristina_da_drive(nome)
print("\n✓ Ripristino completato")

Ripristino file da Drive...

→ S1A_20190708_Orb
✓ S1A_20190708_Orb.dim
✓ S1A_20190708_Orb.data/ (452 MB)

→ S1B_20190702_Orb
✓ S1B_20190702_Orb.dim
✓ S1B_20190702_Orb.data/ (436 MB)

→ S1_Stack
✓ S1_Stack.dim
✓ S1_Stack.data/ (1356 MB)

✓ Ripristino completato


In [ ]:
graph_esd = f"""
<graph id="ESD">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>{OUTPUT_DIR}/S1_Stack.dim</file>
    </parameters>
  </node>
  <node id="Enhanced-Spectral-Diversity">
    <operator>Enhanced-Spectral-Diversity</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <registrationWindowWidth>512</registrationWindowWidth>
      <registrationWindowHeight>512</registrationWindowHeight>
      <searchWindowAccAzimuth>16</searchWindowAccAzimuth>
      <numBlocksPerOverlap>10</numBlocksPerOverlap>
      <esdEstimator>Periodogram</esdEstimator>
      <weightFunction>Inv_Quadratic</weightFunction>
      <spatialAverage>Boxcar</spatialAverage>
      <doNotWriteTargetBands>false</doNotWriteTargetBands>
      <useSuppliedRangeShift>false</useSuppliedRangeShift>
      <overallRangeShift>0.0</overallRangeShift>
      <useSuppliedAzimuthShift>false</useSuppliedAzimuthShift>
      <overallAzimuthShift>0.0</overallAzimuthShift>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Enhanced-Spectral-Diversity"/>
    </sources>
    <parameters>
      <file>{OUTPUT_DIR}/S1_Stack_ESD</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>
"""

graph_path = f'{OUTPUT_DIR}/graphs/esd.xml'
with

SyntaxError: invalid syntax (2266145332.py, line 45)

In [ ]:
graph_esd_xml = """<graph id="ESD">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Stack.dim</file>
    </parameters>
  </node>
  <node id="Enhanced-Spectral-Diversity">
    <operator>Enhanced-Spectral-Diversity</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <registrationWindowWidth>512</registrationWindowWidth>
      <registrationWindowHeight>512</registrationWindowHeight>
      <searchWindowAccAzimuth>16</searchWindowAccAzimuth>
      <numBlocksPerOverlap>10</numBlocksPerOverlap>
      <esdEstimator>Periodogram</esdEstimator>
      <weightFunction>Inv_Quadratic</weightFunction>
      <spatialAverage>Boxcar</spatialAverage>
      <doNotWriteTargetBands>false</doNotWriteTargetBands>
      <useSuppliedRangeShift>false</useSuppliedRangeShift>
      <overallRangeShift>0.0</overallRangeShift>
      <useSuppliedAzimuthShift>false</useSuppliedAzimuthShift>
      <overallAzimuthShift>0.0</overallAzimuthShift>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Enhanced-Spectral-Diversity"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Stack_ESD</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

# Sostituisce il placeholder con il percorso reale
graph_esd_xml = graph_esd_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

# Salva su file
graph_path = f'{OUTPUT_DIR}/graphs/esd.xml'
with open(graph_path, 'w') as f:
    f.write(graph_esd_xml)
print(f"✓ Graph salvato: {graph_path}")

# Esegui
run_gpt(graph_path, "Enhanced Spectral Diversity (ESD)")

✓ Graph salvato: /content/output/graphs/esd.xml

Avvio: Enhanced Spectral Diversity (ESD)

✗ ERRORE in: Enhanced Spectral Diversity (ESD)
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: GDAL not found on system. Internal GDAL 3.2.1 from distribution will be used. (f0)
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Currently installed 9.0.0, available is 13.0.0.
Please visit http://step.esa.int

INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.hsqldb.persist.Logger: dataFileCache open start

Error: [NodeId: Enhanced-Spectral-Diversity] Operator 'SpectralDiversityOp': Unknown element 'registrationWindowWidth'



False

In [ ]:
graph_esd_xml = """<graph id="ESD">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Stack.dim</file>
    </parameters>
  </node>
  <node id="Enhanced-Spectral-Diversity">
    <operator>Enhanced-Spectral-Diversity</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Enhanced-Spectral-Diversity"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Stack_ESD</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_esd_xml = graph_esd_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/esd.xml'
with open(graph_path, 'w') as f:
    f.write(graph_esd_xml)
print(f"✓ Graph salvato")

run_gpt(graph_path, "Enhanced Spectral Diversity (ESD)")

✓ Graph salvato

Avvio: Enhanced Spectral Diversity (ESD)
INFO: org.esa.s1tbx.sentinel1.gpf.SpectralDiversityOp: Shifts written to file: /root/.snap/var/log/IW2_range_shifts.json
INFO: org.esa.s1tbx.sentinel1.gpf.SpectralDiversityOp: Shifts written to file: /root/.snap/var/log/IW2_azimuth_shifts.json

✓ Completato: Enhanced Spectral Diversity (ESD)


True

In [ ]:
salva_su_drive('S1_Stack_ESD')

✓ S1_Stack_ESD.dim
✓ S1_Stack_ESD.data/


In [ ]:
graph_ifg_xml = """<graph id="Interferogram">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Stack_ESD.dim</file>
    </parameters>
  </node>
  <node id="Interferogram">
    <operator>Interferogram</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <subtractFlatEarthPhase>true</subtractFlatEarthPhase>
      <degreeOfFlatEarthPolynomial>5</degreeOfFlatEarthPolynomial>
      <numberOfFlatEarthEstimationPoints>501</numberOfFlatEarthEstimationPoints>
      <orbitDegree>3</orbitDegree>
      <includeCoherence>true</includeCoherence>
      <cohWinAz>3</cohWinAz>
      <cohWinRg>10</cohWinRg>
      <subtractTopographicPhase>false</subtractTopographicPhase>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Interferogram"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_ifg_xml = graph_ifg_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/interferogram.xml'
with open(graph_path, 'w') as f:
    f.write(graph_ifg_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Interferogram Formation + Coherence")

✓ Graph salvato

Avvio: Interferogram Formation + Coherence

✗ ERRORE in: Interferogram Formation + Coherence
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: GDAL not found on system. Internal GDAL 3.2.1 from distribution will be used. (f0)
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.hsqldb.persist.Logger: dataFileCache open start

Error: [NodeId: Interferogram] Operator 'InterferogramOp': Unknown element 'degreeOfFlatEarthPolynomial'



False

In [ ]:
graph_ifg_xml = """<graph id="Interferogram">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Stack_ESD.dim</file>
    </parameters>
  </node>
  <node id="Interferogram">
    <operator>Interferogram</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <subtractFlatEarthPhase>true</subtractFlatEarthPhase>
      <includeCoherence>true</includeCoherence>
      <subtractTopographicPhase>false</subtractTopographicPhase>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Interferogram"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_ifg_xml = graph_ifg_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/interferogram.xml'
with open(graph_path, 'w') as f:
    f.write(graph_ifg_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Interferogram Formation + Coherence")

✓ Graph salvato

Avvio: Interferogram Formation + Coherence

✓ Completato: Interferogram Formation + Coherence


True

In [ ]:
salva_su_drive('S1_Ifg')

✓ S1_Ifg.dim
✓ S1_Ifg.data/


In [ ]:
graph_deburst_xml = """<graph id="Deburst">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg.dim</file>
    </parameters>
  </node>
  <node id="TOPSAR-Deburst">
    <operator>TOPSAR-Deburst</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <selectedPolarisations>VV</selectedPolarisations>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="TOPSAR-Deburst"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg_Deb</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_deburst_xml = graph_deburst_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/deburst.xml'
with open(graph_path, 'w') as f:
    f.write(graph_deburst_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "TOPS Deburst")

✓ Graph salvato

Avvio: TOPS Deburst

✓ Completato: TOPS Deburst


True

In [ ]:
salva_su_drive('S1_Ifg_Deb')

✓ S1_Ifg_Deb.dim
✓ S1_Ifg_Deb.data/


In [ ]:
graph_goldstein_xml = """<graph id="GoldsteinFilter">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg_Deb.dim</file>
    </parameters>
  </node>
  <node id="GoldsteinPhaseFiltering">
    <operator>GoldsteinPhaseFiltering</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <alpha>1.0</alpha>
      <FFTSizeString>64</FFTSizeString>
      <windowSizeString>3</windowSizeString>
      <useCoherenceMask>false</useCoherenceMask>
      <coherenceThreshold>0.2</coherenceThreshold>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="GoldsteinPhaseFiltering"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg_Deb_Flt</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_goldstein_xml = graph_goldstein_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/goldstein.xml'
with open(graph_path, 'w') as f:
    f.write(graph_goldstein_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Goldstein Phase Filtering")

✓ Graph salvato

Avvio: Goldstein Phase Filtering

✓ Completato: Goldstein Phase Filtering


True

In [ ]:
salva_su_drive('S1_Ifg_Deb_Flt')

✓ S1_Ifg_Deb_Flt.dim
✓ S1_Ifg_Deb_Flt.data/


In [ ]:
graph_subset_xml = """<graph id="Subset">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg_Deb_Flt.dim</file>
    </parameters>
  </node>
  <node id="Subset">
    <operator>Subset</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <region>1000,50,24000,2700</region>
      <copyMetadata>true</copyMetadata>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Subset"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg_Deb_Flt_Sub</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_subset_xml = graph_subset_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/subset.xml'
with open(graph_path, 'w') as f:
    f.write(graph_subset_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Subset")

✓ Graph salvato

Avvio: Subset

✓ Completato: Subset


True

In [ ]:
salva_su_drive('S1_Ifg_Deb_Flt_Sub')

✓ S1_Ifg_Deb_Flt_Sub.dim
✓ S1_Ifg_Deb_Flt_Sub.data/


In [ ]:
graph_snaphu_export_xml = """<graph id="SnaphuExport">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Ifg_Deb_Flt_Sub.dim</file>
    </parameters>
  </node>
  <node id="SnaphuExport">
    <operator>SnaphuExport</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <statCostMode>TOPO</statCostMode>
      <initMethod>MCF</initMethod>
      <numberOfTileRows>10</numberOfTileRows>
      <numberOfTileCols>10</numberOfTileCols>
      <numberOfProcessors>4</numberOfProcessors>
      <rowOverlap>200</rowOverlap>
      <colOverlap>200</colOverlap>
      <tileCostThreshold>500</tileCostThreshold>
      <targetFolder>OUTPUT_DIR_PLACEHOLDER/snaphu_export</targetFolder>
    </parameters>
  </node>
</graph>"""

graph_snaphu_export_xml = graph_snaphu_export_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/snaphu_export.xml'
with open(graph_path, 'w') as f:
    f.write(graph_snaphu_export_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Snaphu Export")

✓ Graph salvato

Avvio: Snaphu Export

✓ Completato: Snaphu Export


True

In [ ]:
import os

export_dir = f'{OUTPUT_DIR}/snaphu_export'
print("Contenuto cartella snaphu_export:")
for root, dirs, files in os.walk(export_dir):
    level = root.replace(export_dir, '').count(os.sep)
    indent = '  ' * level
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024*1024)
        print(f"{indent}{f}  ({size_mb:.1f} MB)")

Contenuto cartella snaphu_export:
  UnwPhase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.hdr  (0.0 MB)
  coh_IW2_VV_08Jul2019_02Jul2019.snaphu.img  (247.2 MB)
  coh_IW2_VV_08Jul2019_02Jul2019.snaphu.hdr  (0.0 MB)
  Phase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.img  (247.2 MB)
  Phase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.hdr  (0.0 MB)
  snaphu.conf  (0.0 MB)


In [ ]:
with open(f'{OUTPUT_DIR}/snaphu_export/snaphu.conf', 'r') as f:
    print(f.read())

FileNotFoundError: [Errno 2] No such file or directory: '/content/output/snaphu_export/snaphu.conf'

In [ ]:
with open(f'{OUTPUT_DIR}/snaphu_export/snaphu.conf', 'r') as f:
    print(f.read())

FileNotFoundError: [Errno 2] No such file or directory: '/content/output/snaphu_export/snaphu.conf'

In [ ]:
import os

# Cerca snaphu.conf in tutto /content/output
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        if 'snaphu' in f.lower() or f.endswith('.conf'):
            print(os.path.join(root, f))

/content/output/graphs/snaphu_export.xml
/content/output/snaphu_export/S1_Ifg_Deb_Flt_Sub/UnwPhase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.hdr
/content/output/snaphu_export/S1_Ifg_Deb_Flt_Sub/coh_IW2_VV_08Jul2019_02Jul2019.snaphu.img
/content/output/snaphu_export/S1_Ifg_Deb_Flt_Sub/coh_IW2_VV_08Jul2019_02Jul2019.snaphu.hdr
/content/output/snaphu_export/S1_Ifg_Deb_Flt_Sub/Phase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.img
/content/output/snaphu_export/S1_Ifg_Deb_Flt_Sub/Phase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.hdr
/content/output/snaphu_export/S1_Ifg_Deb_Flt_Sub/snaphu.conf


In [ ]:
export_dir = f'{OUTPUT_DIR}/snaphu_export/S1_Ifg_Deb_Flt_Sub'

with open(f'{export_dir}/snaphu.conf', 'r') as f:
    print(f.read())

# CONFIG FOR SNAPHU
# ---------------------------------------------------------------- 
# Created by SNAP software on: 02:56:59 26/05/2026
#
# Command to call snaphu:
# 
#       snaphu -f snaphu.conf Phase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.img 24000

#########################
# Unwrapping parameters #
#########################

STATCOSTMODE 	 TOPO 
INITMETHOD  	 MCF 
VERBOSE 	 TRUE 

###############
# Input files #
###############

CORRFILE 		coh_IW2_VV_08Jul2019_02Jul2019.snaphu.img

################
# Output files #
################

OUTFILE 		UnwPhase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.img
LOGFILE 		snaphu.log

################
# File formats #
################

INFILEFORMAT 	FLOAT_DATA
CORRFILEFORMAT 	FLOAT_DATA
OUTFILEFORMAT 	FLOAT_DATA

###############################
# SAR and geometry parameters #
###############################

TRANSMITMODE 	REPEATPASS

ORBITRADIUS 	7070449.73
EARTHRADIUS 	6369407.155

LAMBDA 			0.0554658

BASELINE 		190.038
BASELINEANGLE_RAD 	0.101

N

In [ ]:
%%bash
cd /content/output/snaphu_export/S1_Ifg_Deb_Flt_Sub

echo "Avvio Phase Unwrapping con snaphu..."
snaphu -f snaphu.conf Phase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.img 24000
echo "✓ Unwrapping completato"

Avvio Phase Unwrapping con snaphu...

snaphu v2.0.5
27 parameters input from file snaphu.conf (84 lines total)
Logging run-time parameters to file snaphu.log
Creating temporary directory snaphu_tiles_12855
Unwrapping tile at row 0, column 0 (pid 12864)
Unwrapping tile at row 0, column 1 (pid 12869)
Unwrapping tile at row 0, column 2 (pid 12874)
Unwrapping tile at row 0, column 3 (pid 12875)
Unwrapping tile at row 0, column 4 (pid 13270)
Unwrapping tile at row 0, column 5 (pid 13275)
Unwrapping tile at row 0, column 6 (pid 13316)
Unwrapping tile at row 0, column 7 (pid 13367)
Unwrapping tile at row 0, column 8 (pid 13750)
Unwrapping tile at row 0, column 9 (pid 13783)
Unwrapping tile at row 1, column 0 (pid 13824)
Unwrapping tile at row 1, column 1 (pid 13995)
Unwrapping tile at row 1, column 2 (pid 14134)
Unwrapping tile at row 1, column 3 (pid 14207)
Unwrapping tile at row 1, column 4 (pid 14318)
Unwrapping tile at row 1, column 5 (pid 14455)
Unwrapping tile at row 1, column 6 (pid 14

520 incremental costs clipped to avoid overflow (0.011%)
468 incremental costs clipped to avoid overflow (0.010%)
468 incremental costs clipped to avoid overflow (0.010%)
470 incremental costs clipped to avoid overflow (0.010%)


In [ ]:
unwrapped_file = f'{OUTPUT_DIR}/snaphu_export/S1_Ifg_Deb_Flt_Sub/UnwPhase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.hdr'
wrapped_file = f'{OUTPUT_DIR}/S1_Ifg_Deb_Flt_Sub.dim'

graph_snaphu_import_xml = f"""<graph id="SnaphuImport">
  <version>1.0</version>
  <node id="Read-Phase">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>{wrapped_file}</file>
    </parameters>
  </node>
  <node id="Read-Unwrapped-Phase">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>{unwrapped_file}</file>
    </parameters>
  </node>
  <node id="SnaphuImport">
    <operator>SnaphuImport</operator>
    <sources>
      <sourceProduct refid="Read-Phase"/>
      <sourceProduct refid="Read-Unwrapped-Phase"/>
    </sources>
    <parameters>
      <doNotKeepWrapped>false</doNotKeepWrapped>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="SnaphuImport"/>
    </sources>
    <parameters>
      <file>{OUTPUT_DIR}/S1_Unw</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_path = f'{OUTPUT_DIR}/graphs/snaphu_import.xml'
with open(graph_path, 'w') as f:
    f.write(graph_snaphu_import_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Snaphu Import")

✓ Graph salvato

Avvio: Snaphu Import

✗ ERRORE in: Snaphu Import
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: GDAL not found on system. Internal GDAL 3.2.1 from distribution will be used. (f0)
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
INFO: org.esa.s2tbx.dataio.gdal.GDALVersion: Internal GDAL 3.2.1 set to be used by SNAP.
INFO: org.hsqldb.persist.Logger: dataFileCache open start

Error: [NodeId: SnaphuImport] Operator 'SnaphuImportOp': Source product not found: S1_Ifg_Deb_Flt_Sub



False

In [ ]:
unwrapped_file = f'{OUTPUT_DIR}/snaphu_export/S1_Ifg_Deb_Flt_Sub/UnwPhase_ifg_IW2_VV_08Jul2019_02Jul2019.snaphu.hdr'
wrapped_file = f'{OUTPUT_DIR}/S1_Ifg_Deb_Flt_Sub.dim'

graph_snaphu_import_xml = f"""<graph id="SnaphuImport">
  <version>1.0</version>
  <node id="ProductSet-Reader">
    <operator>ProductSet-Reader</operator>
    <sources/>
    <parameters>
      <fileList>{wrapped_file},{unwrapped_file}</fileList>
    </parameters>
  </node>
  <node id="SnaphuImport">
    <operator>SnaphuImport</operator>
    <sources>
      <sourceProduct refid="ProductSet-Reader"/>
    </sources>
    <parameters>
      <doNotKeepWrapped>false</doNotKeepWrapped>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="SnaphuImport"/>
    </sources>
    <parameters>
      <file>{OUTPUT_DIR}/S1_Unw</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_path = f'{OUTPUT_DIR}/graphs/snaphu_import.xml'
with open(graph_path, 'w') as f:
    f.write(graph_snaphu_import_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Snaphu Import")

✓ Graph salvato

Avvio: Snaphu Import

✓ Completato: Snaphu Import


True

In [ ]:
salva_su_drive('S1_Unw')

✓ S1_Unw.dim
✓ S1_Unw.data/


In [ ]:
graph_p2e_xml = """<graph id="PhaseToElevation">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Unw.dim</file>
    </parameters>
  </node>
  <node id="PhaseToElevation">
    <operator>PhaseToElevation</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <demName>SRTM 1Sec HGT</demName>
      <demResamplingMethod>BILINEAR_INTERPOLATION</demResamplingMethod>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="PhaseToElevation"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Elevation</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_p2e_xml = graph_p2e_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/phase_to_elevation.xml'
with open(graph_path, 'w') as f:
    f.write(graph_p2e_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Phase to Elevation")

✓ Graph salvato

Avvio: Phase to Elevation

✓ Completato: Phase to Elevation


True

In [ ]:
salva_su_drive('S1_Elevation')

✓ S1_Elevation.dim
✓ S1_Elevation.data/


In [ ]:
graph_tc_xml = """<graph id="TerrainCorrection">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_Elevation.dim</file>
    </parameters>
  </node>
  <node id="Terrain-Correction">
    <operator>Terrain-Correction</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <demName>SRTM 1Sec HGT</demName>
      <demResamplingMethod>BILINEAR_INTERPOLATION</demResamplingMethod>
      <imgResamplingMethod>BILINEAR_INTERPOLATION</imgResamplingMethod>
      <mapProjection>WGS84(DD)</mapProjection>
      <saveDEM>true</saveDEM>
      <saveLatLon>false</saveLatLon>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Terrain-Correction"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_DEM_Final</file>
      <formatName>BEAM-DIMAP</formatName>
    </parameters>
  </node>
</graph>"""

graph_tc_xml = graph_tc_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/terrain_correction.xml'
with open(graph_path, 'w') as f:
    f.write(graph_tc_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Terrain Correction")

✓ Graph salvato

Avvio: Terrain Correction

✓ Completato: Terrain Correction


True

In [ ]:
salva_su_drive('S1_DEM_Final')
print("\n🎉 DEM salvato su Google Drive!")

✓ S1_DEM_Final.dim
✓ S1_DEM_Final.data/

🎉 DEM salvato su Google Drive!


In [ ]:
graph_export_xml = """<graph id="Export">
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_DEM_Final.dim</file>
    </parameters>
  </node>
  <node id="Write">
    <operator>Write</operator>
    <sources>
      <sourceProduct refid="Read"/>
    </sources>
    <parameters>
      <file>OUTPUT_DIR_PLACEHOLDER/S1_DEM_Final_GeoTiff</file>
      <formatName>GeoTIFF</formatName>
    </parameters>
  </node>
</graph>"""

graph_export_xml = graph_export_xml.replace('OUTPUT_DIR_PLACEHOLDER', OUTPUT_DIR)

graph_path = f'{OUTPUT_DIR}/graphs/export_geotiff.xml'
with open(graph_path, 'w') as f:
    f.write(graph_export_xml)
print("✓ Graph salvato")

run_gpt(graph_path, "Export GeoTiff")

✓ Graph salvato

Avvio: Export GeoTiff

✓ Completato: Export GeoTiff


True

In [ ]:
import shutil
import os

src = f'{OUTPUT_DIR}/S1_DEM_Final_GeoTiff.tif'
dst = f'{DRIVE_DIR}/S1_DEM_Final.tif'

# A volte SNAP salva con nome diverso, cerchiamolo
for f in os.listdir(OUTPUT_DIR):
    if 'GeoTiff' in f and f.endswith('.tif'):
        src = f'{OUTPUT_DIR}/{f}'
        break

if os.path.exists(src):
    shutil.copy2(src, dst)
    size = os.path.getsize(dst) / (1024*1024)
    print(f"✓ GeoTiff salvato su Drive ({size:.0f} MB)")
    print(f"  Percorso: {dst}")
else:
    print("GeoTiff non trovato, controlla il contenuto di OUTPUT_DIR:")
    for f in os.listdir(OUTPUT_DIR):
        print(f"  {f}")

✓ GeoTiff salvato su Drive (252 MB)
  Percorso: /content/drive/MyDrive/S1_InSAR_DEM/S1_DEM_Final.tif
